In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()



In [5]:
import sys
sys.path.append("../")

In [6]:
from module1.rag_helper import RAGBase

In [7]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [8]:

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [9]:
vector_assistant.rag("the program has already begun, can I still sign up?")

Response(id='resp_0a2bfd691fbf9534006a353aaa72fc819ca31660237a13f556', created_at=1781873322.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a2bfd691fbf9534006a353aab2310819caf279c451f50a18e', content=[ResponseOutputText(annotations=[], text='Yes, you can still sign up and start learning. If you want a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781873325.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_summary=None,

In [10]:
vs_index.close()